# Cell-type stratified regional analysis

Integrates Tangram deconvolution with regional BRICHOS vs PBS analysis.

**Prerequisite:** Run `notebooks/00/tangram/12_tangram.ipynb` first to generate `data/ST_BRICHOS_with_tangram.h5ad`.

**Key questions:**
1. Does cell-type composition differ between BRICHOS and PBS per region?
2. Are BRICHOS treatment effects driven by specific cell types (e.g., microglia/immune)?
3. Do PIG genes show cell-type-specific regional responses?

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '../../utils')

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from analysis_utils import plot_spatial_clusters_per_sample


In [ ]:
# Load Tangram-annotated data
import os

tangram_path = '../../data/ST_BRICHOS_with_tangram.h5ad'
base_path = '../../data/ST_BRICHOS_region_subcluster.h5ad'

if os.path.exists(tangram_path):
    adata = sc.read_h5ad(tangram_path)
    print(f'Loaded Tangram-annotated data: {adata.n_obs} spots, {adata.n_vars} genes')
    print(f'Cell types: {adata.obs["tangram_top_label"].value_counts()}')
else:
    print(f'ERROR: {tangram_path} not found.')
    print('Run notebooks/00/tangram/12_tangram.ipynb first to generate Tangram predictions.')
    adata = sc.read_h5ad(base_path)
    print(f'Loaded base data without Tangram: {adata.n_obs} spots')


In [ ]:
# Spatial visualization of Tangram cell-type predictions
if 'tangram_top_label' in adata.obs.columns:
    plot_spatial_clusters_per_sample(adata, color='tangram_top_label', figsize=(20, 15))


## 1. Cell-type composition: BRICHOS vs PBS per region

In [ ]:
# Cell-type composition per (treatment × region)
if 'tangram_top_label' not in adata.obs.columns:
    raise ValueError('Tangram annotations not found. Run Tangram notebook first.')

# Subset to PBS + BRICHOS
ad_trt = adata[adata.obs['treatment'].isin(['PBS', 'BRICHOS'])].copy()

# Compute composition per sample × region
comp_df = (
    ad_trt.obs
    .groupby(['sample_id', 'treatment', 're_annotation_regions', 'tangram_top_label'])
    .size()
    .rename('n_spots')
    .reset_index()
)

# Normalize to proportions within each (sample, region)
totals = comp_df.groupby(['sample_id', 're_annotation_regions'])['n_spots'].transform('sum')
comp_df['proportion'] = comp_df['n_spots'] / totals

print(f'Composition table: {len(comp_df)} rows')
comp_df.head(10)


In [ ]:
# Stacked bar: cell-type composition per treatment × region
regions = ad_trt.obs['re_annotation_regions'].unique()
n_regions = len(regions)
ncols = 3
nrows = int(np.ceil(n_regions / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.ravel(axes)

for ax, region in zip(axes, sorted(regions)):
    sub = comp_df[comp_df['re_annotation_regions'] == region]
    pivot = sub.pivot_table(
        index='treatment', columns='tangram_top_label',
        values='proportion', aggfunc='mean'
    ).fillna(0)
    pivot.plot(kind='bar', stacked=True, ax=ax, legend=False)
    ax.set_title(region, fontsize=10)
    ax.set_ylabel('Proportion')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)

# Hide unused axes
for ax in axes[len(regions):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Cell type', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.suptitle('Cell-type composition: PBS vs BRICHOS per region', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig('../../results/figures/celltype_composition_by_region_treatment.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Test: are cell-type proportions different between PBS and BRICHOS per region?
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

results = []
cell_types = comp_df['tangram_top_label'].unique()

for region in sorted(regions):
    for ct in cell_types:
        sub = comp_df[(comp_df['re_annotation_regions'] == region) & (comp_df['tangram_top_label'] == ct)]
        pbs = sub.loc[sub['treatment'] == 'PBS', 'proportion'].values
        bri = sub.loc[sub['treatment'] == 'BRICHOS', 'proportion'].values
        if len(pbs) < 2 or len(bri) < 2:
            continue
        stat, pval = mannwhitneyu(pbs, bri, alternative='two-sided')
        results.append({
            'region': region, 'cell_type': ct,
            'mean_PBS': pbs.mean(), 'mean_BRICHOS': bri.mean(),
            'delta': bri.mean() - pbs.mean(),
            'pval': pval,
        })

comp_test = pd.DataFrame(results)
if len(comp_test) > 0:
    comp_test['padj'] = multipletests(comp_test['pval'], method='fdr_bh')[1]
    sig = comp_test[comp_test['padj'] < 0.1].sort_values('pval')
    print(f'{len(sig)} significant composition shifts (padj < 0.1):')
    display(sig)
    comp_test.to_csv('../../results/tables/celltype_composition_tests.csv', index=False)
else:
    print('Not enough samples for statistical testing')


## 2. PIG score by cell type and treatment

Are plaque-induced genes specifically modulated in immune/microglial spots?

In [ ]:
# PIG gene list
pig_genes = ['Abi3', 'Apoe', 'Axl', 'B2m', 'C1qa', 'C1qb', 'C1qc', 'C4b',
    'Ccl6', 'Cd63', 'Cd68', 'Cd9', 'Clu', 'Csf1', 'Cst7', 'Ctsd', 'Ctss',
    'Ctsz', 'Cx3cr1', 'Cyba', 'Fcer1g', 'Fth1', 'Gnas', 'Gpnmb', 'Grn',
    'Gusb', 'H2-D1', 'H2-K1', 'Hexa', 'Hexb', 'Hif1a', 'Itgax', 'Lpl',
    'Ly86', 'Lyz2', 'P2ry12', 'Selplg', 'Serpine2', 'Sparc', 'Spi1',
    'Spp1', 'Trem2', 'Tyrobp']

# Score PIG module if not already present
if 'plaque_score' not in ad_trt.obs.columns:
    pig_present = [g for g in pig_genes if g in ad_trt.var_names]
    sc.tl.score_genes(ad_trt, pig_present, score_name='plaque_score')

# PIG score by cell type × treatment × region
pig_df = ad_trt.obs[['sample_id', 'treatment', 're_annotation_regions', 'tangram_top_label', 'plaque_score']].copy()

# Focus on cell types likely relevant to plaque response
# Show all cell types but highlight immune
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(
    data=pig_df, x='tangram_top_label', y='plaque_score',
    hue='treatment', hue_order=['PBS', 'BRICHOS'],
    palette={'PBS': '#d73027', 'BRICHOS': '#4575b4'},
    fliersize=1, linewidth=0.8, ax=ax,
)
ax.set_xlabel('Predicted cell type')
ax.set_ylabel('PIG score')
ax.set_title('Plaque-induced gene score by predicted cell type and treatment')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig('../../results/figures/PIG_score_by_celltype_treatment.pdf', bbox_inches='tight')
plt.show()


## 3. Cell-type-specific BRICHOS effect per region

For key cell types (Immune, Astro-Epen, Vascular), compare PIG scores PBS vs BRICHOS across regions.

In [ ]:
# PIG score: PBS vs BRICHOS per region, stratified by cell type
# Aggregate to sample level first (proper statistical unit)
pig_sample = (
    pig_df
    .groupby(['sample_id', 'treatment', 're_annotation_regions', 'tangram_top_label'])
    ['plaque_score']
    .mean()
    .reset_index()
)

# Focus on cell types with enough representation
ct_counts = pig_sample['tangram_top_label'].value_counts()
major_cts = ct_counts[ct_counts >= 10].index.tolist()

# Identify immune-related cell types (adjust names based on your Tangram output)
immune_related = [ct for ct in major_cts if any(kw in ct.lower() for kw in ['immune', 'micro', 'astro', 'vascular', 'oligo'])]
if not immune_related:
    immune_related = major_cts[:6]  # fallback: top 6 cell types

print(f'Analyzing cell types: {immune_related}')

n_cts = len(immune_related)
fig, axes = plt.subplots(1, n_cts, figsize=(5 * n_cts, 5), sharey=True)
if n_cts == 1:
    axes = [axes]

for ax, ct in zip(axes, immune_related):
    sub = pig_sample[pig_sample['tangram_top_label'] == ct]
    sns.barplot(
        data=sub, x='re_annotation_regions', y='plaque_score',
        hue='treatment', hue_order=['PBS', 'BRICHOS'],
        palette={'PBS': '#d73027', 'BRICHOS': '#4575b4'},
        capsize=0.05, errwidth=1, ax=ax,
    )
    sns.stripplot(
        data=sub, x='re_annotation_regions', y='plaque_score',
        hue='treatment', hue_order=['PBS', 'BRICHOS'],
        palette={'PBS': '#d73027', 'BRICHOS': '#4575b4'},
        dodge=True, size=3, alpha=0.7, ax=ax,
    )
    ax.set_title(ct, fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=90)
    ax.get_legend().remove()

axes[0].set_ylabel('Mean PIG score (per sample)')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles[:2], labels[:2], title='Treatment', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.suptitle('PIG score by cell type, region, and treatment', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig('../../results/figures/PIG_by_celltype_region_treatment.pdf', bbox_inches='tight')
plt.show()


## 4. Cell-type-weighted pseudobulk DGE (Immune focus)

Use Tangram probabilities as weights for immune-enriched spots to run DGE.

In [ ]:
# Immune-focused regional DGE
# Select spots with high immune probability from Tangram

ct_pred = adata.obsm.get('tangram_ct_pred')
if ct_pred is None:
    print('No tangram_ct_pred in obsm. Run Tangram first.')
else:
    # Find immune column
    if hasattr(ct_pred, 'columns'):
        ct_cols = ct_pred.columns.tolist()
    else:
        ct_cols = [f'CT_{i}' for i in range(ct_pred.shape[1])]
    
    immune_cols = [c for c in ct_cols if 'immune' in c.lower() or 'micro' in c.lower()]
    print(f'All cell type columns: {ct_cols}')
    print(f'Immune-related columns: {immune_cols}')
    
    if immune_cols:
        # Sum immune probabilities
        if hasattr(ct_pred, 'loc'):
            immune_prob = ct_pred[immune_cols].sum(axis=1).values
        else:
            immune_idx = [ct_cols.index(c) for c in immune_cols]
            immune_prob = ct_pred[:, immune_idx].sum(axis=1)
        
        adata.obs['immune_prob'] = immune_prob
        
        # Select immune-high spots (top quartile)
        threshold = np.percentile(immune_prob, 75)
        immune_mask = immune_prob >= threshold
        print(f'Immune-high spots (>= {threshold:.3f}): {immune_mask.sum()} / {len(immune_mask)}')
        
        # Spatial visualization
        plot_spatial_clusters_per_sample(adata, color='immune_prob', figsize=(20, 15))
    else:
        print('No immune column found. Available:', ct_cols)
        print('Adjust column name matching above.')


In [ ]:
# DGE on immune-enriched spots: BRICHOS vs PBS per region
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
from anndata import AnnData as AD

if 'immune_prob' in adata.obs.columns:
    # Subset to immune-high spots and PBS/BRICHOS
    ad_imm = adata[
        (adata.obs['immune_prob'] >= np.percentile(adata.obs['immune_prob'], 75)) &
        (adata.obs['treatment'].isin(['PBS', 'BRICHOS']))
    ].copy()
    
    print(f'Immune-enriched spots for DGE: {ad_imm.n_obs}')
    
    # Pseudobulk per (sample × region) on immune spots only
    from notebooks_04_funcs import make_pseudobulk_by_compartment, run_dge_per_compartment  # noqa
    # Or inline if import doesn't work:
    sys.path.insert(0, '.')
    
    # Reuse function from notebook 04 (copy-paste if needed)
    obs = ad_imm.obs[['sample_id', 'treatment', 're_annotation_regions']].copy().astype(str)
    obs['bulk_id'] = obs['sample_id'] + '|' + obs['treatment'] + '|' + obs['re_annotation_regions']
    
    X = ad_imm.layers['counts'] if 'counts' in ad_imm.layers else ad_imm.X
    if hasattr(X, 'tocsr'):
        X = X.tocsr()
    
    uniq_bulk = pd.Index(np.unique(obs['bulk_id'].values), name='bulk_id')
    codes = pd.Categorical(obs['bulk_id'].values, categories=uniq_bulk).codes
    
    M = np.zeros((ad_imm.n_vars, len(uniq_bulk)), dtype=np.int64)
    for j in range(len(uniq_bulk)):
        idx = np.where(codes == j)[0]
        if len(idx) == 0:
            continue
        Xi = X[idx]
        M[:, j] = Xi.sum(axis=0).A.ravel() if hasattr(Xi, 'A') else np.asarray(Xi.sum(axis=0)).ravel()
    
    meta_imm = (
        uniq_bulk.to_series()
        .str.split('|', expand=True)
        .rename(columns={0: 'sample_id', 1: 'treatment', 2: 're_annotation_regions'})
    )
    meta_imm.index = uniq_bulk
    counts_imm = pd.DataFrame(M, index=ad_imm.var_names, columns=uniq_bulk)
    
    # Run DGE per region: BRICHOS vs PBS
    inference = DefaultInference(n_cpus=8)
    immune_dge = {}
    
    for region in meta_imm['re_annotation_regions'].unique():
        mask = meta_imm['re_annotation_regions'] == region
        meta_sub = meta_imm.loc[mask].copy()
        if meta_sub.shape[0] < 3:
            continue
        cts_sub = counts_imm[meta_sub.index]
        
        ad = AD(X=cts_sub.T.values)
        ad.var_names = cts_sub.index
        ad.obs_names = cts_sub.columns
        ad.obs = meta_sub.copy()
        ad.obs['treatment'] = pd.Categorical(ad.obs['treatment'], categories=['PBS', 'BRICHOS'])
        
        try:
            dds = DeseqDataSet(adata=ad, design_factors=['treatment'], refit_cooks=True, inference=inference)
            dds.deseq2()
            stat = DeseqStats(dds, contrast=['treatment', 'BRICHOS', 'PBS'], inference=inference)
            stat.summary()
            immune_dge[region] = stat.results_df
            n_sig = (stat.results_df['padj'] < 0.05).sum()
            print(f'{region}: {n_sig} DE genes in immune-enriched spots')
        except Exception as e:
            print(f'{region}: failed — {e}')
    
    # Save
    import os
    os.makedirs('../../results/tables', exist_ok=True)
    for region, res in immune_dge.items():
        fname = region.replace(' ', '_').replace('/', '_')
        res.to_csv(f'../../results/tables/DGE_immune_BRICHOS_vs_PBS_{fname}.csv')
    print(f'\nSaved {len(immune_dge)} immune-enriched DGE tables')
else:
    print('No immune_prob column. Run cell above first.')
